# 76. 고정 RF+노이즈 제거 오토인코더 앙상블
75번최종수치를보기전두모델의학습내부정상상위5%경계결과를OR결합하도록선언했다.74번global RF/ET뷰와75번denoisingAE의평가구간별예측이같은키·정답인지검사한다. 서로의바깥예측을다시학습에쓰는stacking은아니다. 알려진개발자료를재사용한탐색이라독립새표본성적은아니다.


In [ ]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
a=pd.read_csv(tables/'gas_normal_autoencoder_predictions.csv',parse_dates=['TimeStamp'])
b=pd.read_csv(tables/'gas_product_normalized_joint_predictions.csv',parse_dates=['TimeStamp'])
a=a[(a.model=='denoising_normal_AE')&(a.budget==.05)]
b=b[(b.model=='product_normalized_global')&(b.budget==.05)]
key=['product','TimeStamp','block']
g=a.merge(b[key+['actual','prediction']],on=key,validate='one_to_one',suffixes=('_ae','_rf'))
assert (g.actual_ae==g.actual_rf).all()
g['prediction']=(g.prediction_ae.astype(bool)|g.prediction_rf.astype(bool)).astype(int)
rows=[]
for product,q in g.groupby('product'):
 assert not q.TimeStamp.duplicated().any()
 tn,fp,fn,tp=confusion_matrix(q.actual_ae,q.prediction,labels=[0,1]).ravel()
 rows.append(dict(product=product,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),recall=tp/(tp+fn),F1=2*tp/(2*tp+fp+fn)))
g.to_csv(tables/'gas_fixed_RF_AE_union_predictions.csv',index=False,encoding='utf-8-sig')
m=pd.DataFrame(rows);m.to_csv(tables/'gas_fixed_RF_AE_union_metrics.csv',index=False,encoding='utf-8-sig')
print('Predeclared0.05 RF OR denoisingAE',m.to_string(index=False))
print('Known-development out-of-fold combination, not fresh independent validation.')
